In [1]:
%pip install seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("Calculated path:", parent_dir)
sys.path.append(parent_dir)

from src.data_validation import validate_input
from src.preprocessing import prepare_sales_data
from src.forecasting import forecast_future
from src.evaluation import evaluate_predictions
from src.llm_explainer import explain_forecast

Calculated path: C:\Users\user\ai-sales-forecasting-team


ModuleNotFoundError: No module named 'src.data_validation'

# Technical Documentation & Data Dictionary
## 1. Project Overview
This module implements a sales forecasting pipeline to predict monthly sales performance using historical actual sales data.
## 2. Data Dictionary

| Field Name | Data Type | Description |
| :--- | :--- | :--- |
| `Month` | String / Object | Time period of observation (e.g., Jan, Feb) |
| `Actual Sales` | Integer / Float | Historical actual sales figures recorded |
| `Predicted Sales` | Integer / Float | Model-generated sales forecast output |

## 3. Methodology & Workflow
- **Data Ingestion:** Load historical sales figures into Pandas DataFrames.
- **Model Evaluation:** Utilize regression metrics (`MSE`, `MAE`, `R2 Score`) to measure forecast accuracy.
- **Visualization:** Plot comparative line charts tracking actual vs. predicted trends using Matplotlib/Seaborn.

In [5]:
# 1. Input Parameters
BRANCH_ID = "B001"
HORIZON = 7

# 2. Load and Validate Data (with path check & fallback)
csv_path = os.path.join(parent_dir, "data", "sample_sales.csv") if 'parent_dir' in globals() else "../data/sample_sales.csv"

if not os.path.exists(csv_path):
    df_raw = pd.DataFrame({
        "date": pd.date_range(start="2026-08-01", periods=14, freq="D"),
        "branch_id": ["B001"] * 14,
        "sales": [120, 135, 150, 140, 160, 175, 180, 165, 170, 190, 205, 210, 200, 220]
    })
else:
    try:
        df_raw = pd.read_csv(csv_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df_raw = pd.read_csv(csv_path, encoding='latin1')

# Validate input
is_valid = validate_input(df_raw)
if is_valid is False:
    raise ValueError("Input data validation failed.")

# Preprocess data
df_clean = prepare_sales_data(df_raw)

# 3. Generate Predictions
forecast_values = forecast_future(df_clean, HORIZON)
print(f"--- {HORIZON}-Day Forecast for Branch {BRANCH_ID} ---")
print(forecast_values)

# 4. Plot Forecast
plt.figure(figsize=(10, 4))
plt.plot(df_clean['date'].tail(14), df_clean['sales'].tail(14), label="Historical Sales", marker='o')

# Generate explicit date range for the predicted output list
forecast_dates = pd.date_range(start=pd.to_datetime(df_clean['date'].iloc[-1]) + pd.Timedelta(days=1), periods=HORIZON, freq='D')
plt.plot(forecast_dates, forecast_values, label="Forecasted Sales", marker='s', linestyle='--')

plt.title(f"AI Sales Forecasting Demo - Branch {BRANCH_ID}")
plt.xlabel("Date")
plt.ylabel("Sales ($)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 5. AI Explanation
question = "Which day has the highest expected sales?"
try:
    explanation = explain_forecast(forecast_values, metrics={}, question=question)
    print("\nAI Explanation:")
    print(explanation)
except Exception as e:
    print(f"\nAPI Fallback Active: {e}")

NameError: name 'validate_input' is not defined